In [0]:
-- All CTEs replaced with pre-materialized tables from home_luca_bolognesi.forecasting_tool:
--   blockers, use_case_pipeline_changes, asq_summary, monthly_projection, usecases_filtered
-- ae_email filter applied on the usecases_filtered join; BU/region filters in uco_view WHERE.
-- Note: date-dependent columns (days_to_go_live, hygiene_rules, slippage_risk, etc.) reflect
--   pipeline run time, not real-time current_date().

-- Note: monthly pivot column aliases are prefixed with m_ to avoid digit-starting identifier issues
-- when referencing them via table alias (e.g. mp.m_2025_02 instead of mp.`2025_02`)
with monthly_projection_pivoted as (
  select usecase_id
  , coalesce(m_2025_02, 0) as m_2025_02, coalesce(m_2025_03, 0) as m_2025_03, coalesce(m_2025_04, 0) as m_2025_04, coalesce(m_2025_05, 0) as m_2025_05, coalesce(m_2025_06, 0) as m_2025_06, coalesce(m_2025_07, 0) as m_2025_07
  , coalesce(m_2025_08, 0) as m_2025_08, coalesce(m_2025_09, 0) as m_2025_09, coalesce(m_2025_10, 0) as m_2025_10, coalesce(m_2025_11, 0) as m_2025_11, coalesce(m_2025_12, 0) as m_2025_12, coalesce(m_2026_01, 0) as m_2026_01
  , coalesce(m_2026_02, 0) as m_2026_02, coalesce(m_2026_03, 0) as m_2026_03, coalesce(m_2026_04, 0) as m_2026_04, coalesce(m_2026_05, 0) as m_2026_05, coalesce(m_2026_06, 0) as m_2026_06, coalesce(m_2026_07, 0) as m_2026_07
  , coalesce(m_2026_08, 0) as m_2026_08, coalesce(m_2026_09, 0) as m_2026_09, coalesce(m_2026_10, 0) as m_2026_10, coalesce(m_2026_11, 0) as m_2026_11, coalesce(m_2026_12, 0) as m_2026_12, coalesce(m_2027_01, 0) as m_2027_01
  from 
  (
    select usecase_id, m, ramping_dbus from home_luca_bolognesi.forecasting_tool.monthly_projection_by_usecase
  )
  pivot (sum(ramping_dbus) AS dbus
      for m in (
        '2025-02-01' as m_2025_02,
        '2025-03-01' as m_2025_03,
        '2025-04-01' as m_2025_04,
        '2025-05-01' as m_2025_05,
        '2025-06-01' as m_2025_06,
        '2025-07-01' as m_2025_07,
        '2025-08-01' as m_2025_08,
        '2025-09-01' as m_2025_09,
        '2025-10-01' as m_2025_10,
        '2025-11-01' as m_2025_11,
        '2025-12-01' as m_2025_12,
        '2026-01-01' as m_2026_01,
        '2026-02-01' as m_2026_02,
        '2026-03-01' as m_2026_03,
        '2026-04-01' as m_2026_04,
        '2026-05-01' as m_2026_05,
        '2026-06-01' as m_2026_06,
        '2026-07-01' as m_2026_07,
        '2026-08-01' as m_2026_08,
        '2026-09-01' as m_2026_09,
        '2026-10-01' as m_2026_10,
        '2026-11-01' as m_2026_11,
        '2026-12-01' as m_2026_12,
        '2027-01-01' as m_2027_01
      )
  )
),

forecast_quarterly_projection_pivoted as (
  select usecase_id
  , coalesce(FY26_Q1, 0) FY26_Q1
  , coalesce(FY26_Q2, 0) FY26_Q2
  , coalesce(FY26_Q3, 0) FY26_Q3
  , coalesce(FY26_Q4, 0) FY26_Q4
  , coalesce(FY27_Q1, 0) FY27_Q1
  , coalesce(FY27_Q2, 0) FY27_Q2
  , coalesce(FY27_Q3, 0) FY27_Q3
  , coalesce(FY27_Q4, 0) FY27_Q4
  from 
  (
    select usecase_id, fq, sum(quarterly_incremental_dbus) as quarterly_incremental_dbus 
    from home_luca_bolognesi.forecasting_tool.monthly_projection_by_usecase
    group by usecase_id, fq
  )
  pivot (sum(quarterly_incremental_dbus) AS dbus
      for fq in (
        'FY26-Q1' as FY26_Q1,
        'FY26-Q2' as FY26_Q2,
        'FY26-Q3' as FY26_Q3,
        'FY26-Q4' as FY26_Q4,
        'FY27-Q1' as FY27_Q1,
        'FY27-Q2' as FY27_Q2,
        'FY27-Q3' as FY27_Q3,
        'FY27-Q4' as FY27_Q4
      )
  )
),

uco_view as (
  select c.sales_subregion_level_1, c.sales_subregion_level_2, c.sales_subregion_level_3, c.account_name, c.deployable_account_name, c.account_executive, c.solution_architect, c.dsa_user_name as dsa, c.arr_band, c.usecase_id, c.usecase_name 
    , c.target_onboarding_date, b.target_onboarding_date_fq, c.target_live_date, b.target_live_date_fq, c.target_cloud, c.is_migration_usecase
    , nullif(trim(c.migration_source_platform), '') as migration_source_platform_adj
    , c.is_incremental, b.is_keytechwin
    , c.stage, c.stage_number, c.stage_name_ui, c.days_in_stage, c.days_in_validating, c.days_in_scoping, c.days_in_evaluating, c.days_in_confirming, c.days_in_onboarding
    , c.usecase_description, c.demand_plan_next_steps, c.implementation_notes
    , c.implementation_partner_name, c.has_ps_project, c.use_case_product_enriched
    , b.estimated_monthly_dollar_dbus, c.estimated_monthly_dollar_dbus_weighted, b.total_ramping_days
    , c.estimated_quarterly_dollar_dbus, c.estimated_quarterly_dollar_dbus_weighted
    , b.days_in_stage_bucket, b.usecase_url, b.eval_doc_link, b.onboarding_doc_link, b.implementation_status, b.num_of_blockers, b.blocked_count, b.friction_count, b.blocker_details
    , b.stage_advanced, b.stage_change_description, b.target_date_pulled_foreward, b.target_date_change_description, b.target_live_date_diff_days, b.amount_increased, b.amount_change_description, b.change_amount, b.change_type_label
    , b.manager_notes
    , qp.FY26_Q1, qp.FY26_Q2, qp.FY26_Q3, qp.FY26_Q4, qp.FY27_Q1, qp.FY27_Q2, qp.FY27_Q3, qp.FY27_Q4
    , mp.m_2025_02, mp.m_2025_03, mp.m_2025_04, mp.m_2025_05, mp.m_2025_06, mp.m_2025_07, mp.m_2025_08, mp.m_2025_09, mp.m_2025_10, mp.m_2025_11, mp.m_2025_12
    , mp.m_2026_01, mp.m_2026_02, mp.m_2026_03, mp.m_2026_04, mp.m_2026_05, mp.m_2026_06, mp.m_2026_07, mp.m_2026_08, mp.m_2026_09, mp.m_2026_10, mp.m_2026_11, mp.m_2026_12, mp.m_2027_01
    , b.days_to_go_live, b.days_to_onboarding
    , b.hygiene_rules, b.has_hygiene_issues, b.slippage_risk, b.has_onboarding_slippage_risk, b.has_go_live_slippage_risk, b.onboarding_slippage_details, b.go_live_slippage_details
    , b.stage_advanced_count
    , b.stage_regressed_count
    , b.live_date_advanced_count
    , b.live_date_regressed_count
    , b.amount_grew_count
    , b.amount_shrank_count
    , asq.ASQ_Summary_HTML

    from gtm_silver.use_case_detail as c
    inner join (
      select *
      from home_luca_bolognesi.forecasting_tool.usecases_filtered
      where ae_email like '%' || :ae_email || '%'
      qualify row_number() over (partition by usecase_id order by user_id) = 1
    ) as b
      on b.usecase_id = c.usecase_id
    left join forecast_quarterly_projection_pivoted as qp
      on qp.usecase_id = c.usecase_id  -- LEFT: pipeline tables are Italy-AE scoped; use cases with external AEs still appear with 0 projections
    left join monthly_projection_pivoted as mp
      on mp.usecase_id = c.usecase_id
    left join home_luca_bolognesi.forecasting_tool.asq_summary as asq
      on asq.usecase_id = c.usecase_id
    where c.Business_Unit = :business_unit
      and c.sales_subregion_level_1 = :region_level_1
      and c.sales_subregion_level_2 = :region_level_2
)

select * from uco_view
